# Production Inference and Model Packaging

## Imports

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## Freeze the Final Constants

In [ ]:
DATA_PATH = "../data/Dataset_Eating_Disorder.csv"
PRODUCTION_DIR = Path("../production")
MODEL_PATH = PRODUCTION_DIR / "behavioral_kmeans_v1.joblib"
METADATA_PATH = PRODUCTION_DIR / "behavioral_model_metadata_v1.json"

FINAL_QUESTIONS = [
    "Days_FearLosingControlOverEating",
    "Days_ExcludedFoodControlShapeOrWeight",
    "Days_TriedLimitFoodControlShapeOrWeight",
    "Days_FollowedRulesControlShapeOrWeight",
    "Days_FeltFat",
    "EatLess_ToPreventWeightGain",
    "EatLess_AfterOvereating",
    "AvoidEveningEating_ToWatchWeight",
    "Eat_WhenAnxious",
    "Eat_WhenThingsGoWrong"
]

SCALE_A_MAPPING = {
    "Never": 0, "Seldom": 1, "Sometimes": 2, "Often": 3, "Very often": 4
}
SCALE_B_MAPPING = {
    "No days": 0, "1-5 days": 1, "6-12 days": 2, "13-15 days": 3, "Every day": 4
}

N_CLUSTERS = 2
RANDOM_STATE = 42
N_INIT = 10
MODEL_VERSION = 'behavioral-kmeans-v1.0.0'

## Load and Prepare the Training Data

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df = df.rename(columns={
    "DesireToBuy _FromSnackBarOrCafe": "DesireToBuy_FromSnackBarOrCafe"
})
df_unique = df.drop_duplicates().copy()

ordinal_mappings = {
    question: (SCALE_B_MAPPING if question.startswith("Days_") else SCALE_A_MAPPING)
    for question in FINAL_QUESTIONS
}

X_final = df_unique[FINAL_QUESTIONS].copy()
for question in FINAL_QUESTIONS:
    X_final[question] = X_final[question].map(ordinal_mappings[question])

assert X_final.shape == (601, 10)
assert X_final.isna().sum().sum() == 0
assert X_final.min().min() == 0 and X_final.max().max() == 4

print("Training shape:", X_final.shape)
print("Missing values:", X_final.isna().sum().sum())
print("Encoded range:", X_final.min().min(), "to", X_final.max().max())

X10 = X_final.to_numpy(dtype=float)

## Train the Final Model

In [ ]:
model = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=N_INIT
)

labels = model.fit_predict(X10)
cluster_sizes = np.bincount(labels)
cluster_center_means = model.cluster_centers_.mean(axis=1)
training_silhouette = silhouette_score(X10, labels)

print("Cluster sizes:", cluster_sizes)
print("Cluster-center means:", cluster_center_means.round(4))
print("Silhouette Score:", round(training_silhouette, 6))

## Semantic Mapping

In [ ]:
higher_cluster = int(cluster_center_means.argmax())
lower_cluster = int(cluster_center_means.argmin())

semantic_mapping = {
    higher_cluster: "higher_concern",
    lower_cluster: "lower_concern"
}

print("Semantic mapping:", semantic_mapping)

## 6. Production Thresholds

In [ ]:
centers = model.cluster_centers_
direction = centers[1] - centers[0]
midpoint = (centers[0] + centers[1]) / 2

signed_boundary_distance = (X10 - midpoint) @ direction / np.linalg.norm(direction)
assignment_strength = np.abs(signed_boundary_distance)
distance_to_centroid = np.linalg.norm(X10 - centers[labels], axis=1)

borderline_threshold = float(np.quantile(assignment_strength, 0.25))
unusual_threshold = float(np.quantile(distance_to_centroid, 0.975))

print("Borderline threshold:", round(borderline_threshold, 6))
print("Unusual-profile threshold:", round(unusual_threshold, 6))

## Build Runtime-compatible Metadata

In [ ]:
metadata = {
    "model_version": MODEL_VERSION,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "algorithm": "KMeans",
    "distance_metric": "euclidean",
    "n_clusters": N_CLUSTERS,
    "random_state": RANDOM_STATE,
    "n_init": N_INIT,
    "training_rows": len(X_final),
    "feature_names": FINAL_QUESTIONS,
    "feature_order": FINAL_QUESTIONS,
    "feature_count": len(FINAL_QUESTIONS),
    "ordinal_mappings": ordinal_mappings,
    "cluster_centers": centers.tolist(),
    "semantic_mapping": {str(raw_id): name for raw_id, name in semantic_mapping.items()},
    "semantic_mapping_rule": (
        "The raw cluster whose center has the higher mean encoded response across all 10 "
        "features is labeled 'higher_concern'; the other is 'lower_concern'. Recomputed at "
        "training time via argmax(cluster_centers_.mean(axis=1)) -- never a hardcoded raw id."
    ),
    "borderline_threshold": borderline_threshold,
    "borderline_threshold_definition": (
        "25th percentile of |signed distance to the KMeans decision boundary| across the "
        "training set. is_borderline = assignment_strength <= borderline_threshold."
    ),
    "unusual_profile_threshold": unusual_threshold,
    "unusual_profile_definition": (
        "97.5th percentile of Euclidean distance to the assigned (nearest) centroid across "
        "the training set. is_unusual_profile = distance_to_assigned_centroid >= threshold. "
        "A soft warning flag only -- it does not reject or change the prediction."
    ),
    "assignment_strength_definition": (
        "abs(signed distance to the KMeans decision boundary). A GEOMETRIC DISTANCE in the "
        "original 10-feature ordinal space, NOT a calibrated probability."
    ),
    "training_silhouette": float(training_silhouette),
    "training_inertia": float(model.inertia_),
    "training_cluster_sizes": {str(i): int(size) for i, size in enumerate(cluster_sizes)},
    "dataset_duplicate_policy": (
        "Trained on Dataset B (601 rows) -- exact full-record duplicates removed."
    ),
    "library_versions": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "joblib": joblib.__version__
    },
    "source_notebook": "notebooks/04_production_inference_and_packaging.ipynb",
    "upstream_artifacts": [],
    "not_a_clinical_diagnostic_tool": True
}

print("Metadata keys:", list(metadata))

## 8. Save the Two Production Files

In [ ]:
PRODUCTION_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(model, MODEL_PATH)
with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Saved model:", MODEL_PATH.resolve(), MODEL_PATH.stat().st_size, "bytes")
print("Saved metadata:", METADATA_PATH.resolve(), METADATA_PATH.stat().st_size, "bytes")

## Validate Both Files Directly

In [ ]:
loaded_kmeans = joblib.load(MODEL_PATH)
with open(METADATA_PATH, encoding="utf-8") as file:
    loaded_metadata = json.load(file)

required_metadata_keys = {
    "model_version", "feature_order", "ordinal_mappings", "semantic_mapping",
    "cluster_centers", "borderline_threshold", "unusual_profile_threshold"
}

assert isinstance(loaded_kmeans, KMeans)
assert loaded_kmeans.n_clusters == 2
assert required_metadata_keys.issubset(loaded_metadata)
assert loaded_metadata["feature_order"] == FINAL_QUESTIONS
assert set(loaded_metadata["ordinal_mappings"]) == set(FINAL_QUESTIONS)
assert loaded_metadata["semantic_mapping"] == {"0": "higher_concern", "1": "lower_concern"}
assert np.allclose(loaded_kmeans.cluster_centers_, loaded_metadata["cluster_centers"])
assert np.array_equal(loaded_kmeans.predict(X10), labels)

print("Loaded object type:", type(loaded_kmeans).__name__)
print("Required metadata keys present:", required_metadata_keys.issubset(loaded_metadata))
print("All 601 predictions match:", np.array_equal(loaded_kmeans.predict(X10), labels))

## Test the Existing ML Runtime

In [ ]:
SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from behavioral_model import BehavioralClusterModel, predict_behavioral_cluster

runtime_model = BehavioralClusterModel.load(MODEL_PATH, METADATA_PATH)

low_profile = {
    question: ("No days" if question.startswith("Days_") else "Never")
    for question in FINAL_QUESTIONS
}
high_profile = {
    question: ("Every day" if question.startswith("Days_") else "Very often")
    for question in FINAL_QUESTIONS
}
mixed_profile = df_unique.iloc[0][FINAL_QUESTIONS].to_dict()

runtime_results = {
    "low": predict_behavioral_cluster(low_profile, model=runtime_model),
    "high": predict_behavioral_cluster(high_profile, model=runtime_model),
    "mixed": predict_behavioral_cluster(mixed_profile, model=runtime_model)
}

runtime_results

## Runtime Consistency and Negative Tests

In [ ]:
runtime_raw_labels = []
for _, row in df_unique[FINAL_QUESTIONS].iterrows():
    runtime_raw_labels.append(runtime_model.predict(row.to_dict())["raw_cluster_id"])

assert np.array_equal(runtime_raw_labels, labels)

missing_profile = dict(low_profile)
missing_profile.pop(FINAL_QUESTIONS[0])

invalid_profile = dict(low_profile)
invalid_profile[FINAL_QUESTIONS[0]] = "Unknown response"

negative_results = {}
for name, answers in {"missing_question": missing_profile, "invalid_response": invalid_profile}.items():
    try:
        runtime_model.predict(answers)
        negative_results[name] = "FAILED: accepted"
    except Exception as error:
        negative_results[name] = f"PASSED: {type(error).__name__}: {error}"

print("Runtime predictions matching training labels:", len(runtime_raw_labels), "/ 601")
negative_results

## Summary

Training samples: 601
Features: 10
Algorithm: K-Means
Clusters: 2
Model: behavioral_kmeans_v1.joblib
Metadata: behavioral_model_metadata_v1.json

Joblib contains the K-Means model itself and JSON contains the necessary Contract for Encoding, Semantic Mapping and Thresholds. These two files can be used directly by the current Runtime and Backend.